Code to prove reducibility

In [1]:
#!/usr/bin/env python
import copy
import itertools
import geogebra_graph

In [4]:
#!/usr/bin/env python
import copy
import itertools
import sage.all
import geogebra_graph

class Configuration:
    """
    A class used to represent a specific configuration we are trying
    
    Attributes:
        num_colors: number of colors we are trying to use to extend the coloring
        file_name: the Geogebra file that contains our configuration
    """
    def __init__(self, num_colors, file_name):
        self.num_colors = num_colors
        self.S = set()
        self.U = set()

        self.G= geogebra_graph.geogebra_to_graph(file_name)
        
        #relabeling graph edges from the geogebra file
        frontier_color = (0,0,255,0.0)                                                      #identifying frontier edges with the color blue
        frontier_edges = [e for e in self.G.edges() if e[2]["color"] == frontier_color]     #list of frontier edges in the graph
        for e in self.G.edges():
            self.G.set_edge_label(e[0],e[1],None)                                           #erasing all the extra information from Geogebra
            
        end_points_frontiers = list(set(sum([list(e[:2])                                    #all vertices incident to the frontier edges
            for e in frontier_edges],[])))
        config_vertices = [v for v in end_points_frontiers if self.G.degree(v) > 1]         #all vertices incident to the frontier edges AND are in the boundary
        
        #establishing the cyclic order of the verticies in the configuration
        pos = self.G.get_pos()
        angle_config_vertices = sorted([(atan2(pos[v][1], pos[v][0]),v) for v in config_vertices])
        cyclic_vertices = [x[1] for x in angle_config_vertices]
        self.degrees = [-1] * len(cyclic_vertices)                                          # degree is the number of frontier edges incident to the boundary
        for i,v in enumerate(cyclic_vertices):
            frontiers_at_v = [e for e in frontier_edges if v in e]
            self.degrees[i] = len(frontiers_at_v)
            for j,e in enumerate(frontiers_at_v):
                self.G.set_edge_label(*(e[:2]), (i,j))

    def draw_graph(self):
        self.G.show()

    def does_extend(self, frontier_coloring):
        """
        Using a MIP to check if the frontier coloring can extend into a coloring for self.G
    
        Args:
            frontier_coloring: A specific coloring that we are testing -- a tuple of frozen sets ({1,},{1,2,3},{0,})
    
        Returns:
            True if the coloring extends and False otherwise
        """
    
        p = MixedIntegerLinearProgram()
        x= p.new_variable(binary=True)
        L= self.G.line_graph()
        for e in self.G.edges(): 
            p.add_constraint(p.sum(x[e,i] for i in range(self.num_colors)) == 1)                #every edge must have one color
        
        for e,f in L.edges(labels=False):
            for i in range(self.num_colors):
                p.add_constraint(x[e,i] + x[f,i] <=1)                                           #adjacent edges must have different colors
        
        for e in self.G.edges():
            if e[2]: #is not none
                tuple_index,color_index = e[2]
                p.add_constraint(x[e, frontier_coloring[tuple_index][color_index] ] == 1)       #considers the established coloring of the frontier edges
        
        p.set_objective(0)
        try:
            p.solve()
            return True
        except Exception as e:
            if "no feasible" not in str(e).lower():
                print(f"An exception occurred: {type(e).__name__}")
                print(f"Exception message: {e}")
            return False
        
    def init_colorings(self):
        """
        Generates a list of all the possible frontier colorings and organizes them into ones that are extendable (S) or not (U)
        """    
        C=list(range(self.num_colors))                                  # list of colors, for convenience
        for frontier_coloring in itertools.product(                     # making all the possible tuple of tuples
            *(
              (tuple(x) for x in itertools.combinations(C,d))
              for d in self.degrees
             )
            ):
            if self.does_extend(frontier_coloring):  
                self.S.add(frontier_coloring)
            else:
                self.U.add(frontier_coloring)
        print(len(self.S), len(self.U))
    
    def perform_Kempe(self, frontier_coloring, i, j):
        """
        Creates the (i,j)-auxiliary graph (vertices and edges) for a specific frontier coloring
        
        Args: 
            frontier_coloring: the specific frontier coloring that we will create the auxiliary for
            i: a color in [1, num_colors]
            j: a color in [1, num_colors]
            
        Returns:
            aux_edges: a list of tuples where each tuple is a hyper edge in the auxiliary graph
            vertex_index: a set of vertices that will be in the auxiliary
        """
        aux_edges = list()
        vertex_index = set()
        degree_two_vertices = set()
        
        # for trailing off Kempe Chains or loops in auxillary graph
        # for k in range(len(frontier_coloring)):
            # precolors_vertex_sees = copy.deepcopy(list(frontier_coloring[k]))
            # isChanged = 0
            # for l in range(len(precolors_vertex_sees)):
                # kempe_switch = copy.deepcopy(frontier_coloring)
                # if precolors_vertex_sees[l] == i:
                    # precolors_vertex_sees[l] =j
                    # isChanged += 1
                # else:
                    # if precolors_vertex_sees[l] == j:
                        # precolors_vertex_sees[l]=i
                        # isChanged += 1
            # precolors_vertex_sees.sort()
            # kempe_switch[k] = tuple(precolors_vertex_sees)
            # kempe_switch = tuple(kempe_switch)
            # if isChanged==1:
                # if kempe_switch not in self.S:
                    # aux_edges.append(tuple((k,)))
            # if isChanged==2:
                # aux_edges.append(tuple((k,)))
    
        #Continue from here
        #gets the set of vertices that contains the colors i and j and the "half-edge degree" it has
        for k in range(len(frontier_coloring)):
            color_index=set()
            degree =0
            for l in range(len(frontier_coloring[k])):
                if frontier_coloring[k][l] == i:
                    color_index.add(l)
                    degree+=1
                if frontier_coloring[k][l] == j:
                    color_index.add(l)
                    degree+=1
            if color_index != set():
                vertex_index.add(k)
                if degree == 2:
                    degree_two_vertices.add(k)
    
        #testing the Kempe Chains to see if it extends (multiple vertices)
        for endpoints in itertools.combinations_with_replacement(vertex_index, 2):
            #print(endpoints)
            testing_vertex_index = copy.deepcopy(vertex_index)
            start = endpoints[0]
            end = endpoints[1]
            test_coloring = copy.deepcopy(frontier_coloring)
            coloring_s = copy.deepcopy(list(frontier_coloring[start]))
            coloring_e = copy.deepcopy(list(frontier_coloring[end]))
            for m in range(len(coloring_s)):
                if coloring_s[m] == i:
                    coloring_s[m] =j
                else:
                    if coloring_s[m] == j:
                        coloring_s[m]= i
            coloring_s.sort()
            test_coloring[start] = tuple(coloring_s)
            if start != end:
                for m in range(len(coloring_e)):
                    if coloring_e[m] == i:
                        coloring_e[m] =j
                    else:
                        if coloring_e[m] == j:
                            coloring_e[m]= i
                coloring_e.sort()
                test_coloring[end] = tuple(coloring_e)
                testing_vertex_index.remove(end)
            testing_vertex_index.remove(start)
            
            if tuple(test_coloring) not in self.S:
                if start != end:  
                    for middle in powerset(degree_two_vertices-set(endpoints)):
                        kempe = set(middle) | set(endpoints)
                        kempe= tuple(sorted(kempe))
                        aux_edges.append(kempe)
                else:
                    aux_edges.append(tuple((start,)))
        
        print(f"for colors {i} and {j} the vertices: {vertex_index} and edges: {aux_edges}")                
        return vertex_index, aux_edges
        
    def check_Crossing(self, edge1, edge2): 
        """
        Checks if a pair of hyperedges in the auxiliary graph are crossing
        
        Args: edge 1, edge 2: the pairs are hyperedges in the auxiliary
        
        Returns: True if the two edges are crossing and False if they are not crossing
        """
        is_crossing = True
        if len(edge1) == 1 or len(edge2)==1: #trailing off Kempe chains are not crossing
            is_crossing = False
        else:
            minimum = min(edge1[0], edge2[0]) #edges of Kempe are in order so just need to test the 1st values for the min
            if minimum in edge1:
                start_edge = copy.deepcopy(edge1)
                test_edge = copy.deepcopy(set(edge2))
            else:
                start_edge = copy.deepcopy(edge2)
                test_edge = copy.deepcopy(set(edge1))
            maximum = max(max(test_edge), max(start_edge))
            start_edge = start_edge + (maximum,)
            for k in range(len(start_edge)-1):
                test_range = set(range(start_edge[k], start_edge[k+1]+1))
                intersection = test_range & test_edge
                if intersection == test_edge:
                    is_crossing = False
        print(f"{edge1} and {edge2} are crossing: {is_crossing}")
        return is_crossing
        
    def check_Matching(self, aux_vertices, aux_edges):
        """
        Uses a MIP to check if there is a non-crossing perfect matching in the auxiliary graph
        
        Args: aux_vertices, aux_edges: the vertices and edges of the auxiliary graph 
        
        Returns: True if there is a non-crossing perfect matching found and False if it was not found
        """
        if not aux_vertices: #if empty
            print(f"For auxiliary a NCPM was found")
            return True
        p = MixedIntegerLinearProgram()
        x= p.new_variable(binary=True)
        matching = set()
    
        #if len(aux_edges) != 0 or len(aux_edges) != 1:
            #for i in range(len(aux_edges)):
                #for j in range(i):
                    #crossing = check_Crossing(aux_edges[i], aux_edges[j]) #check crossing
                    #if crossing == True:
                        #print(f'crossing between {aux_edges[i]} and {aux_edges[j]}')
                        #p.add_constraint(x[i]+x[j] <= 1)
        for i,j in itertools.combinations(range(len(aux_edges)),2):
            if self.check_Crossing(aux_edges[i], aux_edges[j]) == True:
                #print(f'crossing between {aux_edges[i]} and {aux_edges[j]}')
                p.add_constraint(x[i]+x[j] <= 1)
                        
        #constraints to make sure all vertices are covered
        for v in aux_vertices:
            adj_edges = list()
            for j in range(len(aux_edges)):
                if v in aux_edges[j]:
                    adj_edges.append(j)
            if adj_edges:  #non-empty
                p.add_constraint(p.sum(x[j] for j in adj_edges) ==1)
            else: #if adj_edges is empty then there is no edges in aux that contain v and so no perfect matching
                print(f"For auxiliary a NCPM was NOT found")
                return False
                
        p.set_objective(0)
        try:
            p.solve()
            print(f"For auxiliary a NCPM was found")
            return True
        except Exception as e:
            if "no feasible" not in str(e).lower():
                print(f"An exception occurred: {type(e).__name__}")
                print(f"Exception message: {e}")
                print(f"For auxiliary a NCPM was NOT found")
            return False
            
    def test_all_colors(self, coloring):
        """
        Takes a frontier coloring and for every pair of i, j colors it will create an auxiliary and check to see if the auxiliary has a NCPM
        
        Args: 
            frontier coloring: the specific frontier coloring we are analyzing
            
        Returns:
            True if a NCPM is found and False if a NCPM is not found
        """
        CL = copy.deepcopy(list(coloring))
        for i in range(self.num_colors):
            for j in range(self.num_colors):
                if i < j:
                    aux_vertices, aux_edges= self.perform_Kempe(CL, i, j)
                    #print(f'{aux}')
                    #print(f'{aux_vertices=},{aux_edges=} for {i} and {j}')
                    matching = self.check_Matching(aux_vertices,aux_edges)
                    if matching == False:
                        #print(f'No non-crossing perfect matching found for colors {i} and {j}')
                        return False
                        #print(f'Found a non-crossing perfect matching auxilliary for colors {i} and {j}')
        return True
    
    def test_reducibility(self):
        """
        Implementing all the components together to see if we can move frontier colorings from U to S
        """
        while True:
            print(f'Starting while loop, {len(self.U)=}')
            remove_from_U = set()
            for coloring in self.U:
                if self.test_all_colors(coloring) == False:
                    #self.S.add(coloring)
                    remove_from_U.add(coloring)
                    clean_output = tuple(next(iter(s)) for s in coloring)
                    print(f'Frontier coloring removed, {clean_output}')
            if remove_from_U: #non-empty
                self.U = self.U - remove_from_U
                self.S = self.S.union(remove_from_U)
                if not self.U: #U is empty
                    print(f'U is empty, therefore it is reducible. YAY!')
                    break
            else:
                print(f'Nothing removed from U, {len(self.U)=}')
                break
            #print(UL[i])
            #print(test_all_colors(UL[i]))
            
    def find_equiv_classes(self, UL):
        equivalence_classes=[]
        for colornum,coloring in enumerate(UL):
            #print(f"{colornum=} {coloring=}")
            for orbit in equivalence_classes:
                if coloring in orbit:
                    break
            else:
                print(f"Adding a new class, {len(equivalence_classes)=}, {coloring=}")
                orbit=set()          
                for i,pi in enumerate(itertools.permutations(range(6))):
                    image=tuple(
                           tuple(sorted([pi[z] for z in x]))
                           for x in coloring)
                    orbit.add(image)
                    # if image not in U:
                    #     print(f"{pi=}",type(pi))
                    #     print(f"{image=}", image in U)
                    #if i>3: break
                equivalence_classes.append(orbit)

    def is_reducible(self):
        """
        Implementing the program to test if it is reducible
        """
        #Testing if configuration is reducible
        self.init_colorings()
        # print("Not Extendable colorings:")
        # for i in self.U:
            # clean_output = tuple(next(iter(s)) for s in i)
            # print(clean_output)
        self.test_reducibility()
        if self.U: #non-empty
            UL = sorted(list(self.U))
            #self.find_equiv_classes(UL)
        else:
            print("YAY! It's reducible")

In [5]:
config = Configuration(3, "P23322.ggb")
config.init_colorings()
coloring = ((2,), (2,), (2,), (0,), (1,))
config.test_all_colors(coloring)

156 87
for colors 0 and 1 the vertices: {3, 4} and edges: [(3, 4)]
For auxiliary a NCPM was found
for colors 0 and 2 the vertices: {0, 1, 2, 3} and edges: [(0, 1), (1,), (2, 3)]
(0, 1) and (1,) are crossing: False
(0, 1) and (2, 3) are crossing: False
(1,) and (2, 3) are crossing: False
For auxiliary a NCPM was found
for colors 1 and 2 the vertices: {0, 1, 2, 4} and edges: [(0, 2), (1,), (1, 4), (4,)]
(0, 2) and (1,) are crossing: False
(0, 2) and (1, 4) are crossing: True
(0, 2) and (4,) are crossing: False
(1,) and (1, 4) are crossing: False
(1,) and (4,) are crossing: False
(1, 4) and (4,) are crossing: False
For auxiliary a NCPM was found


True

In [ ]:
def does_extend(self, self.G, coloring):
    """
    Using a MIP to check if the frontier coloring can extend into a coloring for self.G

    Args:
        self: Configuration object
        self.G: Graph inputted by user from GeoGebra
        coloring: A specific coloring that we are testing -- a tuple of tuples ((1,),(1,2,3),(0,))

    Returns:
        True if the coloring extends and False otherwise
    """

    p = MixedIntegerLinearProgram()
    x= p.new_variable(binary=True)
    L= self.G.line_graph()
    for e in self.G.edges(): 
        p.add_constraint(p.sum(x[e,i] for i in range(self.num_colors)) == 1) #every edge must have one color
    
    for e,f in L.edges(labels=False):
        for i in range(self.num_colors):
            p.add_constraint(x[e,i] + x[f,i] <=1) #adjacent edges must have different colors
    
    for e in self.G.edges():
        if e[2]: #is not none
            tuple_index,color_index = e[2]
            p.add_constraint(x[e, coloring[tuple_index][color_index] ] == 1) #considers the established coloring of the half-edges
    
    p.set_objective(0)
    try:
        p.solve()
        return True
    except Exception as e:
        if "no feasible" not in str(e).lower():
            print(f"An exception occurred: {type(e).__name__}")
            print(f"Exception message: {e}")
        return False
    
def init_colorings(self):
    """
    Generates a list of all the possible colorings and organizes them into ones that are extendable (S) or not (U)

    Args:
        self: Configuration object
        self.G: Graph inputted by user from GeoGebra
        degrees: describes the degree at each vertex in the graph self.G
    """    
    C=list(range(self.num_colors))  # list of colors, for convenience
    for coloring in itertools.product(
        *(
          (tuple(x) for x in itertools.combinations(C,d))
          for d in degrees
         )
        ):
        if does_extend(self, self.G, coloring):  #extend conditions
            self.S.add(coloring)
        else:
            self.U.add(coloring)
    print(len(self.S), len(self.U))

In [ ]:
class Configuration:
    def __init__(self, num_colors, file_name):
        self.num_colors = num_colors
        self.S = set()
        self.U = set()

        self.G= geogebra_graph.geogebra_to_graph(file_name)
        #relabeling graph edges from the geogebra file
        half_edge_color = (0,0,255,0.0) #blue
        half_edges = [e for e in self.G.edges() if e[2]["color"] == half_edge_color]
        for e in self.G.edges():
            self.G.set_edge_label(e[0],e[1],None) #erasing all the information from Geogebra
            
        end_points_half_edges = list(set(sum([list(e[:2]) 
            for e in half_edges],[])))
        config_vertices = [v for v in end_points_half_edges if self.G.degree(v) > 1]
        pos = self.G.get_pos()
        angle_config_vertices = sorted([(atan2(pos[v][1], pos[v][0]),v) for v in config_vertices])
        cyclic_vertices = [x[1] for x in angle_config_vertices]
        self.degrees = [-1] * len(cyclic_vertices) # degree is the number of frontier edges incident to the boundary
        for i,v in enumerate(cyclic_vertices):
            half_edges_at_v = [e for e in half_edges if v in e]
            self.degrees[i] = len(half_edges_at_v)
            for j,e in enumerate(half_edges_at_v):
                self.G.set_edge_label(*(e[:2]), (i,j))

    def draw_graph(self):
        self.G.show()

    def does_extend(self, precoloring):
        """
        Using a MIP to check if the pre-coloring can extend into a coloring for self.G
    
        Args:
            self: Configuration object
            precoloring: A specific coloring that we are testing -- a tuple of tuples ((1,),(1,2,3),(0,))
    
        Returns:
            True if the coloring extends and False otherwise
        """
    
        p = MixedIntegerLinearProgram()
        x= p.new_variable(binary=True)
        L= self.G.line_graph()
        for e in self.G.edges(): 
            p.add_constraint(p.sum(x[e,i] for i in range(self.num_colors)) == 1) #every edge must have one color
        
        for e,f in L.edges(labels=False):
            for i in range(self.num_colors):
                p.add_constraint(x[e,i] + x[f,i] <=1) #adjacent edges must have different colors
        
        for e in self.G.edges():
            if e[2]: #is not none
                tuple_index,color_index = e[2]
                p.add_constraint(x[e, precoloring[tuple_index][color_index] ] == 1) #considers the established coloring of the half-edges
        
        p.set_objective(0)
        try:
            p.solve()
            return True
        except Exception as e:
            if "no feasible" not in str(e).lower():
                print(f"An exception occurred: {type(e).__name__}")
                print(f"Exception message: {e}")
            return False
        
    def init_colorings(self):
        """
        Generates a list of all the possible colorings and organizes them into ones that are extendable (S) or not (U)
    
        Args:
            self: Configuration object
            self.G: Graph inputted by user from GeoGebra
            degrees: describes the degree at each vertex in the graph self.G
        """    
        C=list(range(self.num_colors))  # list of colors, for convenience
        for precoloring in itertools.product(
            *(
              (tuple(x) for x in itertools.combinations(C,d))
              for d in self.degrees
             )
            ):
            if self.does_extend(precoloring):  #extend conditions
                self.S.add(precoloring)
            else:
                self.U.add(precoloring)
        print(len(self.S), len(self.U))

    S1 = set() #list that does work after one change
    U1 = set() #list that don't work
    
    def perform_Kempe(self, coloring, i, j):
        aux_edges = list()
        vertex_index = set()
        degree_two_vertices = set()
        
        # for trailing off Kempe Chains or loops in auxillary graph
        for k in range(len(coloring)):
            precolors_vertex_sees = copy.deepcopy(list(coloring[k]))
            isChanged = 0
            for l in range(len(precolors_vertex_sees)):
                kempe_switch = copy.deepcopy(coloring)
                if precolors_vertex_sees[l] == i:
                    precolors_vertex_sees[l] =j
                    isChanged += 1
                else:
                    if precolors_vertex_sees[l] == j:
                        precolors_vertex_sees[l]=i
                        isChanged += 1
            precolors_vertex_sees.sort()
            kempe_switch[k] = tuple(precolors_vertex_sees)
            kempe_switch = tuple(kempe_switch)
            if isChanged==1:
                if kempe_switch not in self.S:
                    aux_edges.append(tuple((k,)))
            if isChanged==2:
                aux_edges.append(tuple((k,)))
    
        #Continue from here
        #gets the set of vertices that contains the colors i and j and the "half-edge degree" it has
        for k in range(len(coloring)):
            color_index=set()
            degree =0
            for l in range(len(coloring[k])):
                if coloring[k][l] == i:
                    color_index.add(l)
                    degree+=1
                if coloring[k][l] == j:
                    color_index.add(l)
                    degree+=1
            if color_index != set():
                vertex_index.add(k)
                if degree == 2:
                    degree_two_vertices.add(k)
    
        #testing the Kempe Chains to see if it extends (multiple vertices)
        for endpoints in itertools.combinations(vertex_index, 2):
            testing_vertex_index = copy.deepcopy(vertex_index)
            start = endpoints[0]
            end = endpoints[1]
            test_coloring = copy.deepcopy(coloring)
            coloring_s = copy.deepcopy(list(coloring[start]))
            coloring_e = copy.deepcopy(list(coloring[end]))
            for m in range(len(coloring_s)):
                if coloring_s[m] == i:
                    coloring_s[m] =j
                else:
                    if coloring_s[m] == j:
                        coloring_s[m]= i
            coloring_s.sort()
            for m in range(len(coloring_e)):
                if coloring_e[m] == i:
                    coloring_e[m] =j
                else:
                    if coloring_e[m] == j:
                        coloring_e[m]= i
            coloring_e.sort()
            test_coloring[start] = tuple(coloring_s)
            test_coloring[end] = tuple(coloring_e)
            testing_vertex_index.remove(endpoints[0])
            testing_vertex_index.remove(endpoints[1])
            
            if tuple(test_coloring) in self.S:
                S1.add(tuple(test_coloring))
            else:
                for middle in powerset(degree_two_vertices-set(endpoints)):
                    kempe = set(middle) | set(endpoints)
                    kempe= tuple(sorted(kempe))
                    U1.add(tuple(test_coloring))
                    aux_edges.append(kempe)
        return vertex_index, aux_edges
        
    def check_Crossing(edge1, edge2): 
        is_crossing = True
        if len(edge1) == 1 or len(edge2)==1: #trailing off Kempe chains are not crossing
            is_crossing = False
        else:
            minimum = min(edge1[0], edge2[0]) #edges of Kempe are in order so just need to test the 1st values for the min
            if minimum in edge1:
                start_edge = edge1
                test_edge = set(edge2)
            else:
                start_edge = edge2
                test_edge = set(edge1)
            maximum = max(max(test_edge), max(start_edge))
            start_edge = start_edge + (maximum,)
            #print(f'{start_edge=}, {test_edge=}, {maximum=}')
            for k in range(len(start_edge)-1):
                test_range = set(range(start_edge[k], start_edge[k+1]+1))
                intersection = test_range & test_edge
                #print(f'{k=}, {test_range=}, {intersection=}')
                if intersection == test_edge:
                    is_crossing = False
                
        return is_crossing
    #returns true if there is a matching found and false if not
    def check_Matching(aux_vertices, aux_edges):
        if not aux_vertices: #if empty
            return True
        p = MixedIntegerLinearProgram()
        x= p.new_variable(binary=True)
        matching = set()
    
        #if len(aux_edges) != 0 or len(aux_edges) != 1:
            #for i in range(len(aux_edges)):
                #for j in range(i):
                    #crossing = check_Crossing(aux_edges[i], aux_edges[j]) #check crossing
                    #if crossing == True:
                        #print(f'crossing between {aux_edges[i]} and {aux_edges[j]}')
                        #p.add_constraint(x[i]+x[j] <= 1)
        for i,j in itertools.combinations(range(len(aux_edges)),2):
            if check_Crossing(aux_edges[i], aux_edges[j]) == True:
                #print(f'crossing between {aux_edges[i]} and {aux_edges[j]}')
                p.add_constraint(x[i]+x[j] <= 1)
                        
        #constraints to make sure all vertices are covered
        for v in aux_vertices:
            adj_edges = list()
            for j in range(len(aux_edges)):
                if v in aux_edges[j]:
                    adj_edges.append(j)
            if adj_edges:  #non-empty
                p.add_constraint(p.sum(x[j] for j in adj_edges) ==1)
            else: #if adj_edges is empty then there is no edges in aux that contain v and so no perfect matching
                return False
                
        p.set_objective(0)
        try:
            p.solve()
            return True
        except Exception as e:
            if "no feasible" not in str(e).lower():
                print(f"An exception occurred: {type(e).__name__}")
                print(f"Exception message: {e}")
            return False
    def test_all_colors(self, coloring):
        CL = copy.deepcopy(list(coloring))
        for i in range(self.num_colors):
            for j in range(self.num_colors):
                if i < j:
                    aux_vertices, aux_edges= perform_Kempe(self, CL, i, j)
                    #print(f'{aux}')
                    #print(f'{aux_vertices=},{aux_edges=} for {i} and {j}')
                    matching = check_Matching(aux_vertices,aux_edges)
                    if matching == False:
                        #print(f'No non-crossing perfect matching found for colors {i} and {j}')
                        return False
                        #print(f'Found a non-crossing perfect matching auxilliary for colors {i} and {j}')
        return True
    
    def test_reducibility(self):
        while True:
            print(f'Starting while loop, {len(self.U)=}')
            remove_from_U = set()
            for coloring in self.U:
                if self.test_all_colors(self, coloring) == False:
                    self.S.add(coloring)
                    remove_from_U.add(coloring)
            if remove_from_U: #non-empty
                self.U = self.U - remove_from_U
                if not self.U: #U is empty
                    print(f'U is empty, therefore it is reducible. YAY!')
                    break
            else:
                print(f'Nothing removed from U, {len(self.U)=}')
                break
            #print(UL[i])
            #print(test_all_colors(UL[i]))
    def find_equiv_classes(UL):
        equivalence_classes=[]
        for colornum,coloring in enumerate(UL):
            #print(f"{colornum=} {coloring=}")
            for orbit in equivalence_classes:
                if coloring in orbit:
                    break
            else:
                print(f"Adding a new class, {len(equivalence_classes)=}, {coloring=}")
                orbit=set()          
                for i,pi in enumerate(itertools.permutations(range(6))):
                    image=tuple(
                           tuple(sorted([pi[z] for z in x]))
                           for x in coloring)
                    orbit.add(image)
                    # if image not in U:
                    #     print(f"{pi=}",type(pi))
                    #     print(f"{image=}", image in U)
                    #if i>3: break
                equivalence_classes.append(orbit)

    def is_reducible(self):
        #Testing if configuration is reducible
        self.init_colorings()
        self.test_reducibility()
        #if self.U: #non-empty
            #UL = sorted(list(self.U))
            #find_equiv_classes(UL)
        #else:
            #print("YAY! It's reducible")

In [ ]:
S1 = set() #list that does work after one change
U1 = set() #list that don't work

def perform_Kempe(self, coloring, i, j):
    aux_edges = list()
    vertex_index = set()
    degree_two_vertices = set()
    
    # for trailing off Kempe Chains or loops in auxillary graph
    for k in range(len(coloring)):
        precolors_vertex_sees = copy.deepcopy(list(coloring[k]))
        isChanged = 0
        for l in range(len(precolors_vertex_sees)):
            kempe_switch = copy.deepcopy(coloring)
            if precolors_vertex_sees[l] == i:
                precolors_vertex_sees[l] =j
                isChanged += 1
            else:
                if precolors_vertex_sees[l] == j:
                    precolors_vertex_sees[l]=i
                    isChanged += 1
        precolors_vertex_sees.sort()
        kempe_switch[k] = tuple(precolors_vertex_sees)
        kempe_switch = tuple(kempe_switch)
        if isChanged==1:
            if kempe_switch not in self.S:
                aux_edges.append(tuple((k,)))
        if isChanged==2:
            aux_edges.append(tuple((k,)))

    #Continue from here
    #gets the set of vertices that contains the colors i and j and the "half-edge degree" it has
    for k in range(len(coloring)):
        color_index=set()
        degree =0
        for l in range(len(coloring[k])):
            if coloring[k][l] == i:
                color_index.add(l)
                degree+=1
            if coloring[k][l] == j:
                color_index.add(l)
                degree+=1
        if color_index != set():
            vertex_index.add(k)
            if degree == 2:
                degree_two_vertices.add(k)

    #testing the Kempe Chains to see if it extends (multiple vertices)
    for endpoints in itertools.combinations(vertex_index, 2):
        testing_vertex_index = copy.deepcopy(vertex_index)
        start = endpoints[0]
        end = endpoints[1]
        test_coloring = copy.deepcopy(coloring)
        coloring_s = copy.deepcopy(list(coloring[start]))
        coloring_e = copy.deepcopy(list(coloring[end]))
        for m in range(len(coloring_s)):
            if coloring_s[m] == i:
                coloring_s[m] =j
            else:
                if coloring_s[m] == j:
                    coloring_s[m]= i
        coloring_s.sort()
        for m in range(len(coloring_e)):
            if coloring_e[m] == i:
                coloring_e[m] =j
            else:
                if coloring_e[m] == j:
                    coloring_e[m]= i
        coloring_e.sort()
        test_coloring[start] = tuple(coloring_s)
        test_coloring[end] = tuple(coloring_e)
        testing_vertex_index.remove(endpoints[0])
        testing_vertex_index.remove(endpoints[1])
        
        if tuple(test_coloring) in self.S:
            S1.add(tuple(test_coloring))
        else:
            for middle in powerset(degree_two_vertices-set(endpoints)):
                kempe = set(middle) | set(endpoints)
                kempe= tuple(sorted(kempe))
                U1.add(tuple(test_coloring))
                aux_edges.append(kempe)
    return vertex_index, aux_edges

In [ ]:
def check_Crossing(edge1, edge2): 
    is_crossing = True
    if len(edge1) == 1 or len(edge2)==1: #trailing off Kempe chains are not crossing
        is_crossing = False
    else:
        minimum = min(edge1[0], edge2[0]) #edges of Kempe are in order so just need to test the 1st values for the min
        if minimum in edge1:
            start_edge = edge1
            test_edge = set(edge2)
        else:
            start_edge = edge2
            test_edge = set(edge1)
        maximum = max(max(test_edge), max(start_edge))
        start_edge = start_edge + (maximum,)
        #print(f'{start_edge=}, {test_edge=}, {maximum=}')
        for k in range(len(start_edge)-1):
            test_range = set(range(start_edge[k], start_edge[k+1]+1))
            intersection = test_range & test_edge
            #print(f'{k=}, {test_range=}, {intersection=}')
            if intersection == test_edge:
                is_crossing = False
            
    return is_crossing

In [ ]:
#returns true if there is a matching found and false if not
def check_Matching(aux_vertices, aux_edges):
    if not aux_vertices: #if empty
        return True
    p = MixedIntegerLinearProgram()
    x= p.new_variable(binary=True)
    matching = set()

    #if len(aux_edges) != 0 or len(aux_edges) != 1:
        #for i in range(len(aux_edges)):
            #for j in range(i):
                #crossing = check_Crossing(aux_edges[i], aux_edges[j]) #check crossing
                #if crossing == True:
                    #print(f'crossing between {aux_edges[i]} and {aux_edges[j]}')
                    #p.add_constraint(x[i]+x[j] <= 1)
    for i,j in itertools.combinations(range(len(aux_edges)),2):
        if check_Crossing(aux_edges[i], aux_edges[j]) == True:
            #print(f'crossing between {aux_edges[i]} and {aux_edges[j]}')
            p.add_constraint(x[i]+x[j] <= 1)
                    
    #constraints to make sure all vertices are covered
    for v in aux_vertices:
        adj_edges = list()
        for j in range(len(aux_edges)):
            if v in aux_edges[j]:
                adj_edges.append(j)
        if adj_edges:  #non-empty
            p.add_constraint(p.sum(x[j] for j in adj_edges) ==1)
        else: #if adj_edges is empty then there is no edges in aux that contain v and so no perfect matching
            return False
            
    p.set_objective(0)
    try:
        p.solve()
        return True
    except Exception as e:
        if "no feasible" not in str(e).lower():
            print(f"An exception occurred: {type(e).__name__}")
            print(f"Exception message: {e}")
        return False

In [ ]:
def test_all_colors(self, coloring):
    CL = copy.deepcopy(list(coloring))
    for i in range(self.num_colors):
        for j in range(self.num_colors):
            if i < j:
                aux_vertices, aux_edges= perform_Kempe(self, CL, i, j)
                #print(f'{aux}')
                #print(f'{aux_vertices=},{aux_edges=} for {i} and {j}')
                matching = check_Matching(aux_vertices,aux_edges)
                if matching == False:
                    #print(f'No non-crossing perfect matching found for colors {i} and {j}')
                    return False
                    #print(f'Found a non-crossing perfect matching auxilliary for colors {i} and {j}')
    return True

In [ ]:
def test_reducibility(self):
    while True:
        print(f'Starting while loop, {len(self.U)=}')
        remove_from_U = set()
        for coloring in self.U:
            if test_all_colors(self, coloring) == False:
                self.S.add(coloring)
                remove_from_U.add(coloring)
        if remove_from_U: #non-empty
            self.U = self.U - remove_from_U
            if not self.U: #U is empty
                print(f'U is empty, therefore it is reducible. YAY!')
                break
        else:
            print(f'Nothing removed from U, {len(self.U)=}')
            break
        #print(UL[i])
        #print(test_all_colors(UL[i]))

In [ ]:
def find_equiv_classes(UL):
    equivalence_classes=[]
    for colornum,coloring in enumerate(UL):
        #print(f"{colornum=} {coloring=}")
        for orbit in equivalence_classes:
            if coloring in orbit:
                break
        else:
            print(f"Adding a new class, {len(equivalence_classes)=}, {coloring=}")
            orbit=set()          
            for i,pi in enumerate(itertools.permutations(range(6))):
                image=tuple(
                       tuple(sorted([pi[z] for z in x]))
                       for x in coloring)
                orbit.add(image)
                # if image not in U:
                #     print(f"{pi=}",type(pi))
                #     print(f"{image=}", image in U)
                #if i>3: break
            equivalence_classes.append(orbit)

In [ ]:
def is_config_reducible(file_name, num_colors):

    #Testing if configuration is reducible
    self = Configuration(6, file_name)
    init_colorings(self, self.G, degrees)
    test_reducibility(self)
    if self.U: #non-empty
        UL = sorted(list(self.U))
        find_equiv_classes(UL)
    else:
        print("YAY! It's reducible")

Testing Configurations

In [ ]:
is_config_reducible(geogebra_graph.geogebra_to_graph("chordal4cycle.ggb"), 6)